In [13]:
import pandas as pd
import numpy as np
import scipy as sp
import soundfile as sf
import torch
import torch.nn as nn



In [14]:
df_meta = pd.read_csv("data/train.csv")

print(df_meta.head())

  primary_label secondary_labels type  latitude  longitude scientific_name  \
0       1161364               []   []  -22.7562   -46.8666    Guyalna cuta   
1       1161364               []   []  -22.7558   -46.8700    Guyalna cuta   
2       1161364               []   []  -22.7547   -46.8728    Guyalna cuta   
3       1161364               []   []  -22.7547   -46.8728    Guyalna cuta   
4       1161364               []   []  -22.7426   -46.8985    Guyalna cuta   

    common_name class_name  inat_taxon_id         author   license  rating  \
0  Guyalna cuta    Insecta        1161364  Lucas Barbosa  cc-by-nc     0.0   
1  Guyalna cuta    Insecta        1161364  Lucas Barbosa  cc-by-nc     0.0   
2  Guyalna cuta    Insecta        1161364  Lucas Barbosa  cc-by-nc     0.0   
3  Guyalna cuta    Insecta        1161364  Lucas Barbosa  cc-by-nc     0.0   
4  Guyalna cuta    Insecta        1161364  Lucas Barbosa  cc-by-nc     0.0   

                                                 url          

In [15]:
class DataProcessor:
    import librosa
    def __init__(self, class_name, input_path, df_meta_subset):
        self.class_name = class_name
        self.input_path = input_path
        self.df_meta_subset = df_meta_subset
        self.labels = np.empty(0, dtype=object)
        self.n_mels = 128
        self.n_fft = 2048
        self.hop_length = 512
    

    def _open_resample_melspec(self, file):
        data, samplerate = sf.read(f'{self.input_path}{file}')
        resampled_data = sp.signal.resample(data, samples)
        mel_spec = librosa.feature.melspectrogram(y=resampled_data, sr=samplerate, 
                                                  n_mels=self.n_mels, n_fft=self.n_fft, hop_length=self.hop_length)
        self.labels = np.append(self.labels, self.df_meta_subset[self.df_meta_subset["filename"] == file]['primary_label'].values[0])
        return mel_spec
    
    def process_files(self):
        self.mel_spec_array = np.array([self._open_resample_melspec(file) for file in self.df_meta_subset["filename"]])

    def save_arrays(self):
        np.save(f"{self.class_name}_mel_spec.npy", self.mel_spec_array)
        np.save(f"{self.class_name}_labels.npy", self.labels)

In [16]:
input_path = 'data/train_audio/'
class_type = 'greyel'
df_meta_subset = df_meta[df_meta['filename'].str.contains(class_type)]

greyel_processor = DataProcessor(class_type, input_path, df_meta_subset)
greyel_processor.process_files()
greyel_processor.save_arrays()

size_mb = greyel_processor.mel_spec_array.nbytes / (1024 ** 2)
print(f"{size_mb:.2f} MB")
print(f"Shape of mel spectrogram array: {greyel_processor.mel_spec_array.shape}")
print(f"Shape of labels array: {greyel_processor.labels.shape}")

453.20 MB
Shape of mel spectrogram array: (475, 128, 977)
Shape of labels array: (475,)
